In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType
from pyspark.sql import Row

catalog_name = 'iranian_transactions'

In [0]:
df_gold_transactions = spark.table(f"{catalog_name}.silver.slv_transactions")

In [0]:
#Get all the transactions with the status = success 
df_gold_transactions_success = df_gold_transactions.filter(F.col("status") == "Success")
display(df_gold_transactions_success)

df_gold_transactions_fail=df_gold_transactions.filter(F.col("status")=="Fail")
display(df_gold_transactions_fail)

display(df_gold_transactions)

In [0]:
#Let's find the total amount of transactions for each city
df_gold_transactions.groupBy("city").sum("amount").orderBy(F.col("sum(amount)").desc()).display()  # was missing ()

by_city = df_gold_transactions.groupBy("city") \
    .agg(F.sum("amount").alias("total_amount"), F.sum(F.when(F.col("is_debt"), -F.col("amount")).otherwise(0)).alias("total_debt")) \
    .orderBy(F.col("total_amount").desc())

#let's find the total amount of transactions fir each card_type
df_gold_transactions.groupBy("card_type").sum("amount").orderBy(F.col("sum(amount)").desc()).display()

#let's find the total amount of transactions for each status
df_gold_transactions.groupBy("status").sum("amount").orderBy(F.col("sum(amount)").desc()).display()

#let's get the negative values and store them in a new column with the header, Debt
df_gold_transactions = df_gold_transactions.withColumn("Debt", F.when(F.col("amount") < 0, F.col("amount")*-1).otherwise(0))
display(df_gold_transactions)
#Let's find the total amount of transactions for each city
df_gold_transactions.groupBy("city").sum("amount").orderBy(F.col("sum(amount)").desc()).display


In [0]:
#Find the total number of people that are in debt and the total amount of debt for each city
df_gold_transactions.groupBy("city").sum("Debt").orderBy(F.col("sum(Debt)").desc()).display()

#Find the total number of people that are in debt and the average amount of debt for each city
df_gold_transactions.groupBy("city").avg("Debt").orderBy(F.col("avg(Debt)").desc()).display()


#Find the total number of people that are in debt and the average amount of debt for each city
df_gold_transactions.groupBy("city").count().orderBy(F.col("count").desc()).display()

#Find the average amount of debtors in the city of Tehran with a status of Success
df_gold_transactions.filter(F.col("city") == "Tehran").filter(F.col("status") == "Success").groupBy("city").count().orderBy(F.col("count").desc()).display()


#Find the total number of people that are in debt and the average amount of debt for each card_type
df_gold_transactions.groupBy("card_type").avg("Debt").orderBy(F.col("avg(Debt)").desc()).display()

#Let's find the total number of people that are in debt and the total amount of debt for each card_type
df_gold_transactions.groupBy("card_type").sum("Debt").orderBy(F.col("sum(Debt)").desc()).display()


In [0]:

#Find the average amount of debtors in the city of Tehran with a status of Success
df_gold_transactions.filter(F.col("city") == "Tehran").filter(F.col("status") == "Success").groupBy("city").avg("Debt").orderBy(F.col("avg(Debt)").desc()).display()

df_gold_transactions.filter(F.col("city") == "Tehran").filter(F.col("status") == "Fail").groupBy("city").avg("Debt").orderBy(F.col("avg(Debt)").desc()).display()

In [0]:
print("\nDimension counts:")
print("dim_date     :", spark.table(f"{catalog_name}.gold.dim_date").count())
print("dim_card     :", spark.table(f"{catalog_name}.gold.dim_card").count())
print("dim_location :", spark.table(f"{catalog_name}.gold.dim_location").count())
print("dim_status   :", spark.table(f"{catalog_name}.gold.dim_status").count())
print("dim_time     :", spark.table(f"{catalog_name}.gold.dim_time").count())  # should be 1440

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog_name}.gold.fact_transactions
USING DELTA
AS
SELECT
  monotonically_increasing_id() AS transaction_sk,   -- optional surrogate
  
  t.id AS transaction_id,
  
  -- Foreign keys
  d.date_sk,                                         -- from dim_date (keep this one)
  tm.time_id,
  c.card_id,
  l.location_id,
  s.status_id,
  
  -- Measures
  t.amount,
  coalesce(t.fees, 0)            AS fees,
  coalesce(t.discount, 0)        AS discount,
  coalesce(t.balance_before, 0)  AS balance_before,
  
  -- Useful attributes
  t.is_weekend,
  t._source_file,
  t.ingested_at
FROM {catalog_name}.silver.slv_transactions t

-- Date dimension (you should already have this)
LEFT JOIN {catalog_name}.gold.dim_date d
  ON t.date = d.full_date

-- Time dimension
LEFT JOIN {catalog_name}.gold.dim_time tm
  ON t.hour = tm.hour AND t.minute = tm.minute

-- Card dimension
LEFT JOIN {catalog_name}.gold.dim_card c
  ON t.card_type = c.card_provider

-- Location dimension
LEFT JOIN {catalog_name}.gold.dim_location l
  ON t.city = l.city_name

-- Status dimension
LEFT JOIN {catalog_name}.gold.dim_status s
  ON t.status = s.status_label
""")

In [0]:
#Validation queries

# Row counts
print("Fact rows:", spark.table(f"{catalog_name}.gold.fact_transactions").count())
print("Dim Date:", spark.table(f"{catalog_name}.gold.dim_date").count())
print("Dim Card:", spark.table(f"{catalog_name}.gold.dim_card_type").count())
print("Dim Location:", spark.table(f"{catalog_name}.gold.dim_location").count())
print("Dim Status:", spark.table(f"{catalog_name}.gold.dim_status").count())

# Sample star join
spark.sql(f"""
SELECT 
  d.year, d.month_name, d.day_name,
  c.card_type, c.card_family,
  l.city, l.province,
  s.status_group,
  COUNT(*) AS txn_count,
  SUM(f.amount) AS total_amount
FROM {catalog_name}.gold.fact_transactions f
JOIN {catalog_name}.gold.dim_date d ON f.date_sk = d.date_sk
JOIN {catalog_name}.gold.dim_card_type c ON f.card_type_sk = c.card_type_sk
JOIN {catalog_name}.gold.dim_location l ON f.location_sk = l.location_sk
JOIN {catalog_name}.gold.dim_status s ON f.status_sk = s.status_sk
GROUP BY 1,2,3,4,5,6,7,8
ORDER BY txn_count DESC
LIMIT 20
""").display()

In [0]:
%sql
-- Daily summary by city + card
CREATE OR REPLACE TABLE gold.mart_daily_city_card AS
SELECT 
  d.full_date,
  l.city_name,
  c.card_provider,
  s.status_label,
  COUNT(*) AS txn_count,
  SUM(f.amount) AS total_amount,
  AVG(f.amount) AS avg_amount,
  SUM(f.fees) AS total_fees
FROM gold.fact_transactions f
JOIN gold.dim_date d ON f.date_sk = d.date_sk
JOIN gold.dim_location l ON f.location_id = l.location_id
JOIN gold.dim_card c ON f.card_id = c.card_id
JOIN gold.dim_status s ON f.status_id = s.status_id
GROUP BY 1,2,3,4;

In [0]:

# =====================================================================
# 1. Hourly volume patterns (by time_of_day)
# =====================================================================
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog_name}.gold.mart_hourly_volume
USING DELTA
AS
SELECT
  t.time_of_day,
  t.hour,
  COUNT(*)                                      AS txn_count,
  SUM(f.amount)                                 AS total_amount,
  ROUND(AVG(f.amount), 2)                       AS avg_amount,
  SUM(CASE WHEN s.status_label = 'Success' THEN 1 ELSE 0 END) AS success_count,
  SUM(CASE WHEN s.status_label IN ('Fail', 'Failed') THEN 1 ELSE 0 END) AS fail_count,
  ROUND(
    SUM(CASE WHEN s.status_label = 'Success' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
  ) AS success_rate_pct
FROM {catalog_name}.gold.fact_transactions f
JOIN {catalog_name}.gold.dim_time   t ON f.time_id = t.time_id
JOIN {catalog_name}.gold.dim_status s ON f.status_id = s.status_id
GROUP BY t.time_of_day, t.hour
ORDER BY t.hour
""")

print("✓ mart_hourly_volume created")

# =====================================================================
# 2. Success vs Fail rates by city / card
# =====================================================================
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog_name}.gold.mart_success_fail_by_city_card
USING DELTA
AS
SELECT
  l.city_name,
  c.card_provider,
  COUNT(*)                                      AS total_txns,
  SUM(CASE WHEN s.status_label = 'Success' THEN 1 ELSE 0 END) AS success_count,
  SUM(CASE WHEN s.status_label IN ('Fail', 'Failed') THEN 1 ELSE 0 END) AS fail_count,
  SUM(CASE WHEN s.status_label IN ('Processing', 'Pending') THEN 1 ELSE 0 END) AS in_progress_count,
  ROUND(
    SUM(CASE WHEN s.status_label = 'Success' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
  ) AS success_rate_pct,
  ROUND(
    SUM(CASE WHEN s.status_label IN ('Fail', 'Failed') THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
  ) AS fail_rate_pct,
  ROUND(AVG(f.amount), 2)                       AS avg_amount,
  SUM(f.amount)                                 AS total_amount
FROM {catalog_name}.gold.fact_transactions f
JOIN {catalog_name}.gold.dim_location l ON f.location_id = l.location_id
JOIN {catalog_name}.gold.dim_card     c ON f.card_id = c.card_id
JOIN {catalog_name}.gold.dim_status   s ON f.status_id = s.status_id
GROUP BY l.city_name, c.card_provider
ORDER BY total_txns DESC
""")

print("✓ mart_success_fail_by_city_card created")

# =====================================================================
# 3. Weekend vs Weekday comparison
# =====================================================================
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog_name}.gold.mart_weekend_vs_weekday
USING DELTA
AS
SELECT
  CASE WHEN d.is_weekend THEN 'Weekend' ELSE 'Weekday' END AS day_type,
  COUNT(*)                                      AS txn_count,
  SUM(f.amount)                                 AS total_amount,
  ROUND(AVG(f.amount), 2)                       AS avg_amount,
  SUM(CASE WHEN s.status_label = 'Success' THEN 1 ELSE 0 END) AS success_count,
  SUM(CASE WHEN s.status_label IN ('Fail', 'Failed') THEN 1 ELSE 0 END) AS fail_count,
  ROUND(
    SUM(CASE WHEN s.status_label = 'Success' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
  ) AS success_rate_pct,
  ROUND(
    SUM(CASE WHEN s.status_label IN ('Fail', 'Failed') THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
  ) AS fail_rate_pct
FROM {catalog_name}.gold.fact_transactions f
JOIN {catalog_name}.gold.dim_date   d ON f.date_sk = d.date_sk
JOIN {catalog_name}.gold.dim_status s ON f.status_id = s.status_id
GROUP BY CASE WHEN d.is_weekend THEN 'Weekend' ELSE 'Weekday' END
""")

print("✓ mart_weekend_vs_weekday created")

# =====================================================================
# 4. Top cities by volume and by failure rate
# =====================================================================
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog_name}.gold.mart_top_cities
USING DELTA
AS
SELECT
  l.city_name,
  COUNT(*)                                      AS total_txns,
  SUM(f.amount)                                 AS total_volume,
  ROUND(AVG(f.amount), 2)                       AS avg_amount,
  SUM(CASE WHEN s.status_label = 'Success' THEN 1 ELSE 0 END) AS success_count,
  SUM(CASE WHEN s.status_label IN ('Fail', 'Failed') THEN 1 ELSE 0 END) AS fail_count,
  ROUND(
    SUM(CASE WHEN s.status_label IN ('Fail', 'Failed') THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
  ) AS fail_rate_pct,
  ROUND(
    SUM(CASE WHEN s.status_label = 'Success' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
  ) AS success_rate_pct,
  -- ranking helpers
  RANK() OVER (ORDER BY COUNT(*) DESC)          AS rank_by_volume,
  RANK() OVER (ORDER BY 
    SUM(CASE WHEN s.status_label IN ('Fail', 'Failed') THEN 1 ELSE 0 END) * 100.0 / COUNT(*) DESC
  ) AS rank_by_fail_rate
FROM {catalog_name}.gold.fact_transactions f
JOIN {catalog_name}.gold.dim_location l ON f.location_id = l.location_id
JOIN {catalog_name}.gold.dim_status   s ON f.status_id = s.status_id
GROUP BY l.city_name
""")

print("✓ mart_top_cities created")

Other useful marts:

** Hourly volume patterns (time_of_day)
** Success vs Fail rates by city / card
** Weekend vs weekday comparison
** Top cities by volume or failure rate
